# F1 weekly race predictions — baseline v2

This is the maintained weekly workflow. Edit the configuration cell and run all. The notebook includes its own pipeline, so no separate Python scripts are needed in Colab. CPU is sufficient.

The included CSVs are a historical seed through 2026 round 12. The initial configuration updates completed races through 2026 round 15 (Baku), then trains the fixed baseline model. Prediction is disabled until the next race's starting grid or verified API inputs are available. Data updates replace existing race keys instead of appending duplicates.

For the next prediction, keep `COMPLETED_YEAR` and `COMPLETED_ROUND` at the latest finished race and set `PREDICT_YEAR`/`PREDICT_ROUND` to the next event. If the update and training already succeeded with unchanged data and package versions, set `UPDATE_WEEKEND=False`, `TRAIN_MODEL=False`, `RUN_PREDICTION=True`. If no saved weekly checkpoint exists, train first.

`INPUT_MODE="api"` fetches published qualifying and a verified starting grid. API results can lag or be unavailable; this mode fails closed instead of guessing a grid. `INPUT_MODE="manual"` uses `STARTING_GRID`, `QUALIFYING_ORDER`, optional `QUALIFYING_TIMES`, and circuit metadata supplied in the configuration. Keep the actual qualifying order separate from the penalty-adjusted starting grid. `QUALIFYING_ORDER=[]` explicitly uses the grid as a proxy and is less informative when penalties apply. Drivers starting in the pit lane remain listed in `STARTING_GRID` and must also be named in `PIT_LANE_STARTERS`.

The frozen baseline recipe is used for weekly retraining. Practice-derived features were investigated separately but were not selected as the production model. The model outputs a predicted classification and DNF probabilities; it cannot anticipate individual crashes or incidents.


In [ ]:
# EDIT THIS CELL, THEN RUN ALL.
DATA_DIR = "/content/drive/MyDrive/F1_Data/fastf1_revision"

# Latest completed race: 2026 Azerbaijan (Baku), round 15.
COMPLETED_YEAR = 2026
COMPLETED_ROUND = 15
PREDICT_YEAR = 2026
PREDICT_ROUND = 16

# First run from the included historical CSVs: update and train through R15.
# Prediction is off until the next race's grid is known.
UPDATE_WEEKEND = True
TRAIN_MODEL = True
RUN_PREDICTION = False

# For prediction, use "api" when qualifying and the official grid are published.
# Use "manual" before publication or to test a hypothetical starting grid.
INPUT_MODE = "api"
STARTING_GRID = []       # Manual mode only, P1 first
QUALIFYING_ORDER = []    # Manual mode: true qualifying order, before penalties
QUALIFYING_TIMES = {}    # Optional: {"RUS": {"Q1": "1:43.615", "Q2": "1:43.462", "Q3": "1:42.526"}}
PIT_LANE_STARTERS = []
WITHDRAWN_DRIVERS = []
TEAM_OVERRIDES = {}      # Manual mode: {"DRIVER": "constructor_id"}
EXPECTED_PREDICTION_EVENT = ""  # Optional guard, e.g. "Bahrain"
MANUAL_CIRCUIT_ID = ""        # Required only for manual prediction
MANUAL_EVENT_NAME = ""        # Required only for manual prediction


## Automatic setup

In [ ]:
# Automatic setup. No edits required.
import sys
import subprocess
import importlib.util

requirements = {
    "numpy": "numpy>=1.26", "pandas": "pandas>=2.2",
    "scipy": "scipy>=1.11", "sklearn": "scikit-learn>=1.4",
    "xgboost": "xgboost>=2.0", "joblib": "joblib>=1.3",
}
if INPUT_MODE == "api":
    requirements["fastf1"] = "fastf1==3.8.3"
missing = [package for module, package in requirements.items()
           if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])
    raise RuntimeError("Dependencies installed. Restart the session, then Run all again. Do not delete the runtime.")
if DATA_DIR.startswith("/content/drive/"):
    from google.colab import drive
    drive.mount("/content/drive")
print("Data folder:", DATA_DIR)


## Shared pipeline — no edits needed

In [ ]:
import hashlib
import json
import os
import re
import shutil
import tempfile
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd

DRIVER_KEY = ['Year', 'Round Number', 'Driver Abbr']
TEAM_KEY = ['Year', 'Round Number', 'Team ID']

def frame_fingerprint(frame):
    """Stable across CSV roundtrips that infer integer rather than float dtypes."""
    return hashlib.sha256(frame.to_csv(index=False, float_format='%.12g').encode()).hexdigest()

TEAM_ALIASES = {
    'Red Bull Racing': 'Red Bull', 'Alpine F1 Team': 'Alpine',
    'RB F1 Team': 'Racing Bulls', 'RB': 'Racing Bulls',
    'Cadillac F1 Team': 'Cadillac', 'Alfa Romeo Racing': 'Alfa Romeo',
}
CIRCUIT_ALIASES = {
    'Hangaroring': 'Hungaroring', 'Hungary': 'Hungaroring',
    'Monaco': 'Monte Carlo', 'Barcelona': 'Catalunya', 'Spain': 'Catalunya',
    'Losail': 'Lusail', 'Qatar': 'Lusail', 'Miami Gardens': 'Miami',
    'Montréal': 'Montreal', 'Las Vegas Grand Prix': 'Las Vegas',
    'Netherlands': 'Zandvoort', 'Australia': 'Melbourne',
    'Yas Marina': 'Yas Marina Circuit', 'Marina Bay': 'Singapore',
    'São Paulo': 'Interlagos', 'Sao Paulo': 'Interlagos',
}
DISPLAY_CIRCUITS = {
    'albert_park': 'Melbourne', 'bahrain': 'Sakhir', 'jeddah': 'Jeddah',
    'americas': 'Austin', 'baku': 'Baku', 'catalunya': 'Catalunya',
    'hungaroring': 'Hungaroring', 'monaco': 'Monte Carlo', 'villeneuve': 'Montreal',
    'losail': 'Lusail', 'vegas': 'Las Vegas', 'miami': 'Miami',
    'red_bull_ring': 'Spielberg', 'yas_marina': 'Yas Marina Circuit',
    'spa': 'Spa-Francorchamps', 'sakhir': 'Sakhir Outer Track',
}

def canonical_team(value):
    return TEAM_ALIASES.get(str(value).strip(), str(value).strip())

def canonical_circuit(value):
    return CIRCUIT_ALIASES.get(str(value).strip(), str(value).strip())

def seconds(value):
    if value is None or pd.isna(value) or str(value) in ('', 'nan', 'NaT'):
        return np.nan
    try:
        if hasattr(value, 'total_seconds'):
            return value.total_seconds()
        parts = str(value).split(':')
        return sum(float(v) * 60 ** i for i, v in enumerate(parts[::-1]))
    except (TypeError, ValueError):
        return np.nan

def status_flags(status, laps=0):
    text = str(status).strip().lower()
    dns = text in {'did not start', 'did not qualify', 'did not prequalify'} or (text == 'withdrew' and int(laps) == 0)
    normal = text in {'finished', 'lapped'} or bool(re.fullmatch(r'\+\d+ laps?', text))
    dsq = 'disqual' in text or 'excluded' in text
    return int(not normal and not dns and not dsq), int(dns), int(dsq)

def validate_drivers(drivers, require_complete=True):
    required = DRIVER_KEY + ['Team', 'Team ID', 'Circuit ID', 'Race Position',
                             'Race Points', 'Sprint Points', 'Did Not Finish', 'Did Not Start']
    missing = set(required) - set(drivers)
    if missing:
        raise ValueError(f'Missing driver columns: {sorted(missing)}')
    if drivers.duplicated(DRIVER_KEY).any():
        raise ValueError('Duplicate driver/race keys; reconcile them before training.')
    if drivers[DRIVER_KEY + ['Team ID', 'Circuit ID']].isna().any().any():
        raise ValueError('Missing race or identity keys.')
    for key, race in drivers.groupby(['Year', 'Round Number']):
        pos = pd.to_numeric(race['Race Position'], errors='coerce')
        required_pos = pos[race['Did Not Start'] == 0]
        if require_complete and (required_pos.isna().any() or required_pos.duplicated().any()):
            raise ValueError(f'{key}: finishing classifications are missing or duplicated.')
        if (((pos < 1) | (pos > len(race)) | (pos % 1 != 0)) & pos.notna()).any():
            raise ValueError(f'{key}: invalid finishing position.')
        if not 15 <= len(race) <= 24:
            raise ValueError(f'{key}: suspicious field size {len(race)}.')
        if race['Circuit ID'].nunique() != 1:
            raise ValueError(f'{key}: inconsistent circuit identity.')
    if (drivers[['Race Points', 'Sprint Points']] < 0).any().any():
        raise ValueError('Negative per-session points.')
    return True

def build_constructor_table(drivers, adjustments=None):
    """Rebuild each season from weekend points, keyed by constructor identity.

    Adjustments are optional explicit championship deductions/additions, keyed
    (year, round, team_id). They are applied once, from that round onward.
    Countback uses wins, then seconds, etc. Alphabetical ID is a stable final tie.
    """
    adjustments = adjustments or {}
    data = []
    for year, season in drivers.groupby('Year', sort=True):
        totals, counts, names = {}, {}, {}
        for rnd, race in season.groupby('Round Number', sort=True):
            current = []
            for team_id, team in race.groupby('Team ID'):
                names[team_id] = team.iloc[0]['Team']
                weekend = float(team['Race Points'].sum() + team['Sprint Points'].sum())
                change = float(adjustments.get((int(year), int(rnd), team_id), 0))
                totals[team_id] = totals.get(team_id, 0.) + weekend + change
                counts.setdefault(team_id, np.zeros(24, dtype=int))
                for p in team['Race Position'].dropna().astype(int):
                    counts[team_id][p - 1] += 1
                meta = team.iloc[0]
                current.append({
                    'Year': int(year), 'Round Number': int(rnd),
                    'Circuit Name': meta['Circuit Name'], 'Race Name': meta['Race Name'],
                    'Mean Track Temperature': meta.get('Mean Track Temperature', np.nan),
                    'Rainfall Status': meta.get('Rainfall Status', 'Unknown'),
                    'Team': names[team_id], 'Weekend_Points': weekend,
                    'Team_Cons': names[team_id], 'Total Points': totals[team_id],
                    'Team ID': team_id, 'Circuit ID': meta['Circuit ID'],
                    'Points Adjustment': change,
                })
            standings = sorted(totals, key=lambda tid: (-totals[tid], *(-counts[tid]), tid))
            rank = {tid: i + 1 for i, tid in enumerate(standings)}
            for row in current:
                row['Rank'] = rank[row['Team ID']]
            data.extend(sorted(current, key=lambda row: row['Rank']))
    original = ['Year', 'Round Number', 'Circuit Name', 'Race Name',
                'Mean Track Temperature', 'Rainfall Status', 'Team',
                'Weekend_Points', 'Rank', 'Team_Cons', 'Total Points']
    return pd.DataFrame(data).reindex(columns=original + ['Team ID', 'Circuit ID', 'Points Adjustment'])

def validate_pair(drivers, constructors):
    validate_drivers(drivers)
    if constructors.duplicated(TEAM_KEY).any():
        raise ValueError('Duplicate constructor/race keys.')
    adjustment_map = {(int(r.Year), int(r['Round Number']), r['Team ID']): float(r['Points Adjustment'])
                      for _, r in constructors.iterrows() if float(r.get('Points Adjustment', 0)) != 0}
    expected = build_constructor_table(drivers, adjustment_map)
    check = expected.merge(constructors, on=TEAM_KEY, how='outer', suffixes=('_expected', '_saved'), indicator=True)
    if not check['_merge'].eq('both').all():
        raise ValueError('Driver/constructor race keys do not match.')
    for col in ['Weekend_Points', 'Total Points', 'Rank']:
        if not np.allclose(check[f'{col}_expected'], check[f'{col}_saved'], atol=1e-8):
            raise ValueError(f'Constructor {col} does not reconcile.')
    return True

def save_pair(drivers, constructors, driver_path, team_path):
    """Validate before any write, back up both files, and roll back on failure."""
    validate_pair(drivers, constructors)
    paths = [Path(driver_path), Path(team_path)]
    for path in paths:
        path.parent.mkdir(parents=True, exist_ok=True)
    stamp = pd.Timestamp.now(tz='UTC').strftime('%Y%m%dT%H%M%S%fZ')
    backups, staged = {}, []
    try:
        for path, frame in zip(paths, [drivers, constructors]):
            if path.exists():
                backup = path.parent / 'backups' / f'{path.stem}_{stamp}.csv'
                backup.parent.mkdir(exist_ok=True)
                shutil.copy2(path, backup)
                backups[path] = backup
            fd, temp = tempfile.mkstemp(prefix=path.stem, suffix='.tmp', dir=path.parent)
            os.close(fd)
            frame.to_csv(temp, index=False)
            staged.append(Path(temp))
        for temp, path in zip(staged, paths):
            os.replace(temp, path)
    except Exception:
        for path in paths:
            if path in backups:
                shutil.copy2(backups[path], path)
            elif path.exists():
                path.unlink()
        raise
    finally:
        for temp in staged:
            temp.unlink(missing_ok=True)

def result_frame(race, qualifying=None, sprint=None, original=None, source_url=''):
    """Construct authoritative outcome rows; never infer an R driver's rank."""
    qualifying = qualifying or {}
    sprint = sprint or {}
    qrows = {r['Driver']['code']: r for r in qualifying.get('QualifyingResults', [])}
    srows = {r['Driver']['code']: r for r in sprint.get('SprintResults', [])}
    orig = {} if original is None else {
        key: group.iloc[0].to_dict() for key, group in original.groupby('Driver Abbr')}
    circuit_id = race['Circuit']['circuitId']
    old_circuit = next(iter(orig.values()), {}).get('Circuit Name', race['Circuit']['Location'].get('locality', circuit_id))
    circuit = DISPLAY_CIRCUITS.get(circuit_id, canonical_circuit(old_circuit))
    q1_values = [seconds(r.get('Q1')) for r in qrows.values()]
    q1_min = min((v for v in q1_values if np.isfinite(v)), default=np.nan)
    pole = next((r for r in qrows.values() if str(r.get('position')) == '1'), {})
    pole_time = next((seconds(pole.get(k)) for k in ['Q3', 'Q2', 'Q1'] if np.isfinite(seconds(pole.get(k)))), np.nan)
    records = []
    for result in race.get('Results', []):
        code = result['Driver']['code']
        previous, q, s = orig.get(code, {}), qrows.get(code, {}), srows.get(code, {})
        dnf, dns, dsq = status_flags(result.get('status', 'Unknown'), result.get('laps', 0))
        temp = previous.get('Mean Track Temperature', np.nan)
        placeholder = pd.notna(temp) and float(temp) == 0.
        times = [seconds(q.get(k)) for k in ['Q1', 'Q2', 'Q3']]
        best = min((v for v in times if np.isfinite(v)), default=np.nan)
        session_best = next((seconds(q.get(k)) for k in ['Q3', 'Q2', 'Q1'] if np.isfinite(seconds(q.get(k)))), np.nan)
        row = {
            'Year': int(race['season']), 'Round Number': int(race['round']),
            'Circuit Name': circuit, 'Race Name': race['raceName'],
            'Mean Track Temperature': np.nan if placeholder else temp,
            'Rainfall Status': 'Unknown' if placeholder else previous.get('Rainfall Status', 'Unknown'),
            'Driver Abbr': code, 'Team': canonical_team(result['Constructor']['name']),
            'Driver Number': int(result.get('number', result['Driver'].get('permanentNumber', 0))),
            'Sprint Position': float(s['position']) if s else np.nan,
            'Sprint Points': float(s.get('points', 0)),
            'Quali Position': float(q['position']) if q else np.nan,
            'Race Position': int(result['position']), 'Race Points': float(result['points']),
            'Team ID': result['Constructor']['constructorId'], 'Circuit ID': circuit_id,
            'Driver ID': result['Driver']['driverId'], 'Race Date': race['date'],
            'Grid Position': float(result['grid']) if 'grid' in result else np.nan,
            'Grid Source': 'public race result' if 'grid' in result else 'unknown',
            'Status': result.get('status', 'Unknown'),
            'Classified Position': str(result.get('positionText', result['position'])),
            'Did Not Finish': dnf, 'Did Not Start': dns, 'Disqualified': dsq,
            'Laps Completed': int(result.get('laps', 0)),
            'Q1 Gap Percent': 100 * (seconds(q.get('Q1')) / q1_min - 1) if q1_min > 0 else np.nan,
            'Quali Gap To Pole Percent': 100 * (session_best / pole_time - 1) if pole_time > 0 else np.nan,
            'Quali Best Seconds': best,
            'Practice Pace Delta': previous.get('Practice Pace Delta', np.nan),
            'PreRace Track Temperature': previous.get('PreRace Track Temperature', np.nan),
            'PreRace Rainfall': previous.get('PreRace Rainfall', np.nan),
            'Team Original': previous.get('Team Original', previous.get('Team', '')),
            'Circuit Name Original': previous.get('Circuit Name Original', previous.get('Circuit Name', '')),
            'Race Position Original': previous.get('Race Position Original', previous.get('Race Position', '')),
            'Data Source': source_url,
        }
        records.append(row)
    frame = pd.DataFrame(records)
    if len(frame):
        validate_drivers(frame)
    return frame


import time
import urllib.request

def public_constructor_standings(year, rnd, cache_dir, refresh=False):
    path = Path(cache_dir) / f'{int(year)}_{int(rnd)}_constructor_standings.json'
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists() and not refresh:
        payload = json.loads(path.read_text())
    else:
        url = f'https://api.jolpi.ca/ergast/f1/{int(year)}/{int(rnd)}/constructorStandings.json?limit=100'
        for attempt in range(3):
            try:
                payload = json.load(urllib.request.urlopen(url, timeout=30))
                break
            except Exception:
                if attempt == 2:
                    raise
                time.sleep(2 ** attempt)
        if payload['MRData']['StandingsTable']['StandingsLists']:
            path.write_text(json.dumps(payload))
    tables = payload['MRData']['StandingsTable']['StandingsLists']
    if not tables:
        raise ValueError('Constructor standings are not available yet; no CSVs were changed.')
    table = tables[0]
    if (int(table['season']), int(table['round'])) != (int(year), int(rnd)):
        raise ValueError('Constructor standings refer to a different round.')
    return {r['Constructor']['constructorId']: float(r['points']) for r in table['ConstructorStandings']}

def public_race_session(year, rnd, kind, cache_dir, refresh=False):
    """Small, persistent JSON cache. No full telemetry download is needed."""
    if kind not in {'results', 'qualifying', 'sprint', 'schedule'}:
        raise ValueError('Unsupported session type.')
    cache_dir = Path(cache_dir)
    cache_dir.mkdir(parents=True, exist_ok=True)
    path = cache_dir / f'{int(year)}_{int(rnd)}_{kind}.json'
    suffix = '.json' if kind == 'schedule' else f'/{kind}.json'
    url = f'https://api.jolpi.ca/ergast/f1/{int(year)}/{int(rnd)}{suffix}?limit=100'
    if path.exists() and not refresh:
        payload = json.loads(path.read_text())
    else:
        for attempt in range(3):
            try:
                request = urllib.request.Request(url, headers={'User-Agent': 'F1-ranking-notebook/1.0'})
                payload = json.load(urllib.request.urlopen(request, timeout=30))
                break
            except Exception:
                if attempt == 2:
                    raise
                time.sleep(2 ** attempt)
        # Do not permanently cache empty/uncompleted sessions.
        if payload['MRData']['RaceTable']['Races']:
            path.write_text(json.dumps(payload))
    races = payload['MRData']['RaceTable']['Races']
    if not races:
        return None
    race = races[0]
    if int(race['season']) != int(year) or int(race['round']) != int(rnd):
        raise ValueError('API returned the wrong event.')
    return race

def append_single_race(year, rnd, drivers_csv, constructors_csv, cache_dir, refresh=True):
    old_d, old_t = pd.read_csv(drivers_csv), pd.read_csv(constructors_csv)
    race = public_race_session(year, rnd, 'results', cache_dir, refresh)
    if race is None or not race.get('Results'):
        raise ValueError('Race results are not available. Existing CSVs were not changed.')
    quali = public_race_session(year, rnd, 'qualifying', cache_dir, refresh)
    if quali is None:
        raise ValueError('Qualifying results unavailable. Existing CSVs were not changed.')
    sprint = public_race_session(year, rnd, 'sprint', cache_dir, refresh)
    event = public_race_session(year, rnd, 'schedule', cache_dir, refresh)
    if event is None or ('Sprint' in event and not sprint):
        raise ValueError('Event schedule or scheduled sprint results are missing. Existing CSVs were not changed.')
    same = (old_d.Year == year) & (old_d['Round Number'] == rnd)
    fresh = result_frame(race, quali, sprint, old_d[same],
                        f'https://api.jolpi.ca/ergast/f1/{year}/{rnd}/results.json')
    old_entries = set(old_d.loc[same & (old_d['Did Not Start'] == 0), 'Driver Abbr'])
    if old_entries - set(fresh['Driver Abbr']):
        raise ValueError('The refreshed result omits previously recorded starters. Review before replacing this race.')
    # Preserve separately verified withdrawals that the public result does not list.
    dns = old_d[same & (old_d['Did Not Start'] == 1)]
    dns = dns[~dns['Driver Abbr'].isin(fresh['Driver Abbr'])]
    new_d = pd.concat([old_d[~same], fresh, dns], ignore_index=True).sort_values(DRIVER_KEY).reset_index(drop=True)
    adjustments = {(int(r.Year), int(r['Round Number']), r['Team ID']): float(r['Points Adjustment'])
                   for _, r in old_t.iterrows() if float(r.get('Points Adjustment', 0)) != 0}
    new_t = build_constructor_table(new_d, adjustments)
    official_points = public_constructor_standings(year, rnd, cache_dir, refresh)
    current = new_t[(new_t.Year == year) & (new_t['Round Number'] == rnd)]
    for _, row in current.iterrows():
        tid = row['Team ID']
        if tid not in official_points:
            raise ValueError(f'Constructor {tid} missing from published standings. No files changed.')
        difference = official_points[tid] - float(row['Total Points'])
        if abs(difference) > 1e-8:
            key = (int(year), int(rnd), tid)
            adjustments[key] = adjustments.get(key, 0.) + difference
            print(f'Reconciled championship adjustment {year} R{rnd} {tid}: {difference:+g} points')
    new_t = build_constructor_table(new_d, adjustments)
    save_pair(new_d, new_t, drivers_csv, constructors_csv)
    print(f'Updated {year} R{rnd}: {len(fresh)} result rows. Both originals backed up.')
    return new_d, new_t

def build_entries(drivers, year, rnd, starting_grid, team_overrides=None, qualifying=None):
    """P1-first grid. A rookie must have an explicitly supplied current team ID."""
    if not 15 <= len(starting_grid) <= 24 or len(set(starting_grid)) != len(starting_grid):
        raise ValueError('Supply 15–24 unique driver abbreviations in actual starting-grid order.')
    overrides = team_overrides or {}
    past = drivers[(drivers.Year < year) | ((drivers.Year == year) & (drivers['Round Number'] < rnd))]
    qrows = {r['Driver']['code']: r for r in (qualifying or {}).get('QualifyingResults', [])}
    q1_min = min((seconds(r.get('Q1')) for r in qrows.values() if np.isfinite(seconds(r.get('Q1')))), default=np.nan)
    pole = next((r for r in qrows.values() if str(r.get('position')) == '1'), {})
    pole_time = next((seconds(pole.get(k)) for k in ['Q3', 'Q2', 'Q1'] if np.isfinite(seconds(pole.get(k)))), np.nan)
    rows = []
    for grid, code in enumerate(starting_grid, 1):
        q = qrows.get(code, {})
        history = past[past['Driver Abbr'] == code].sort_values(['Year', 'Round Number'])
        tid = overrides.get(code, {}).get('Team ID')
        if tid is None and q:
            tid = q['Constructor']['constructorId']
        if tid is None and len(history):
            tid = history.iloc[-1]['Team ID']
        if tid is None:
            raise ValueError(f'No current team for rookie {code}; supply team_overrides. No driver is silently dropped.')
        last_segment = next((seconds(q.get(k)) for k in ['Q3', 'Q2', 'Q1'] if np.isfinite(seconds(q.get(k)))), np.nan)
        # Qualifying is not assumed equal to the supplied starting grid.
        rows.append({'Driver Abbr': code, 'Team ID': tid,
            'Grid Position': overrides.get(code, {}).get('Grid Position', grid),
            'Quali Position': float(q['position']) if q else overrides.get(code, {}).get('Quali Position', np.nan),
            'Q1 Gap Percent': 100 * (seconds(q.get('Q1')) / q1_min - 1) if q1_min > 0 else np.nan,
            'Quali Gap To Pole Percent': 100 * (last_segment / pole_time - 1) if pole_time > 0 else np.nan,
            'Practice Pace Delta': np.nan, 'PreRace Track Temperature': np.nan, 'PreRace Rainfall': np.nan})
    return pd.DataFrame(rows)


FP_COLUMNS = ['fp_short_gap', 'fp_long_gap', 'fp_sector1_gap', 'fp_sector2_gap', 'fp_sector3_gap', 'fp_speed_delta', 'fp_clean_laps', 'fp_long_laps', 'fp_track_temp', 'fp_rainfall']
from collections import defaultdict

FEATURE_VERSION = 'f1-ranking-v1.0'
CATEGORICAL_FEATURES = ['circuit_id', 'team_id']
NUMERIC_FEATURES = [
    'grid_pct', 'quali_pct', 'grid_penalty', 'pit_lane_start', 'grid_missing',
    'quali_missing', 'q1_gap', 'quali_gap', 'driver_form', 'driver_quali_form',
    'driver_current_team_form', 'team_form', 'team_quali_form',
    'driver_history_count', 'team_history_count', 'recent_trend',
    'grid_conversion', 'track_form', 'track_history_count',
    'driver_dnf_rate', 'team_dnf_rate', 'teammate_quali_delta',
    'current_teammate_quali_delta', 'team_rank_pct', 'team_points_per_race',
    'expected_pace', 'grid_pace_gap', 'practice_pace_delta',
    'pre_race_temperature', 'pre_race_rain', 'season_round', 'field_size',
]
FEATURE_COLUMNS = NUMERIC_FEATURES + CATEGORICAL_FEATURES

def finite_float(value, fallback=np.nan):
    try:
        value = float(value)
        return value if np.isfinite(value) else fallback
    except (TypeError, ValueError):
        return fallback

def scaled_position(pos, field_size):
    value = finite_float(pos)
    return (value - 1) / max(1, field_size - 1) if np.isfinite(value) and value > 0 else np.nan

def weighted_history(history, key, year, window=7, fallback=.5, season_discount=.4):
    available = [r for r in history if np.isfinite(r.get(key, np.nan))][-window:]
    if not available:
        return fallback
    ages = np.arange(len(available) - 1, -1, -1)
    weights = np.exp2(-ages / max(window / 2, 1)) * np.array([
        season_discount ** max(0, year - r['year']) for r in available])
    # A team/year change reduces the contribution of old history to the prior.
    raw = np.average([r[key] for r in available], weights=weights)
    confidence = min(1., weights.sum() / 2.)
    return float(confidence * raw + (1 - confidence) * fallback)

class FeatureBuilder:
    """One race-at-a-time state; features are generated before outcomes are added."""
    def __init__(self, window=7, season_discount=.4):
        self.window = int(window)
        self.season_discount = float(season_discount)
        self.drivers = defaultdict(list)
        self.teams = defaultdict(list)
        self.standings = {}
        self.last_key = None

    def make_features(self, entries, year, rnd, circuit_id):
        year, rnd = int(year), int(rnd)
        if self.last_key is not None and self.last_key >= (year, rnd):
            raise ValueError('Feature state includes the prediction race or future outcomes.')
        n = len(entries)
        if entries['Driver Abbr'].duplicated().any():
            raise ValueError('Prediction grid contains duplicate drivers.')
        grid_values = pd.to_numeric(entries['Grid Position'], errors='coerce')
        positive = grid_values[grid_values > 0]
        if positive.duplicated().any() or ((positive % 1 != 0) | (positive > n)).any() or (grid_values < 0).any():
            raise ValueError('Grid positions must be unique integers 1..field size; zero means pit lane.')
        rows = []
        for _, entry in entries.iterrows():
            code, tid = str(entry['Driver Abbr']), str(entry['Team ID'])
            history, team_history = self.drivers[code], self.teams[tid]
            mean = lambda h, k, default=.5: weighted_history(
                h, k, year, self.window, default, self.season_discount)
            team_form = mean(team_history, 'finish')
            driver_form = mean(history, 'finish', team_form)
            grid = finite_float(entry.get('Grid Position'))
            quali = finite_float(entry.get('Quali Position'))
            grid_missing = not np.isfinite(grid)
            # Qualifying fallback is explicit. Pit-lane zero is retained as a flag.
            effective_grid = (quali if np.isfinite(quali) else n) if grid_missing else (n if grid == 0 else grid)
            grid_pct = scaled_position(effective_grid, n)
            quali_pct = scaled_position(quali, n)
            recent = [h['finish'] for h in history[-3:] if np.isfinite(h['finish'])]
            trend = float(np.polyfit(np.arange(len(recent)), recent, 1)[0]) if len(recent) > 1 else 0.
            track = [h for h in history if h['circuit'] == circuit_id and year - h['year'] <= 3]
            standings = self.standings.get((year, tid), {'rank_pct': .5, 'points_per_race': 0.})
            team_mates = entries[(entries['Team ID'] == tid) & (entries['Driver Abbr'] != code)]
            mate_quali = pd.to_numeric(team_mates['Quali Position'], errors='coerce').mean()
            pace = .6 * driver_form + .4 * team_form
            dh = history[-20:]
            th = team_history[-10:]
            row = dict(zip(CATEGORICAL_FEATURES, [str(circuit_id), tid]))
            row.update({
                'grid_pct': grid_pct, 'quali_pct': quali_pct,
                'grid_penalty': (effective_grid - quali) / max(1, n - 1) if np.isfinite(quali) else np.nan,
                'pit_lane_start': float(grid == 0), 'grid_missing': float(grid_missing),
                'quali_missing': float(not np.isfinite(quali)),
                'q1_gap': finite_float(entry.get('Q1 Gap Percent')),
                'quali_gap': finite_float(entry.get('Quali Gap To Pole Percent')),
                'driver_form': driver_form, 'driver_quali_form': mean(history, 'quali'),
                'driver_current_team_form': mean([h for h in history if h['team'] == tid], 'finish', team_form),
                'team_form': team_form, 'team_quali_form': mean(team_history, 'quali'),
                'driver_history_count': min(len(history), 30), 'team_history_count': min(len(team_history), 30),
                'recent_trend': trend, 'grid_conversion': mean(history, 'conversion'),
                'track_form': mean(track, 'finish', driver_form), 'track_history_count': len(track),
                'driver_dnf_rate': (2. + sum(h['dnf'] for h in dh)) / (20 + len(dh)),
                'team_dnf_rate': (2. + sum(h['dnf'] for h in th)) / (20 + len(th)),
                'teammate_quali_delta': mean(history, 'mate_delta', 0.),
                'current_teammate_quali_delta': (quali - mate_quali) / max(1, n - 1),
                'team_rank_pct': standings['rank_pct'], 'team_points_per_race': standings['points_per_race'],
                'expected_pace': pace, 'grid_pace_gap': grid_pct - pace,
                'practice_pace_delta': finite_float(entry.get('Practice Pace Delta')),
                'pre_race_temperature': finite_float(entry.get('PreRace Track Temperature')),
                'pre_race_rain': finite_float(entry.get('PreRace Rainfall')),
                'season_round': rnd, 'field_size': n,
            })
            rows.append(row)
        return pd.DataFrame(rows, columns=FEATURE_COLUMNS)

    def observe(self, race, constructor_round):
        year, rnd = int(race.iloc[0]['Year']), int(race.iloc[0]['Round Number'])
        if self.last_key is not None and self.last_key >= (year, rnd):
            raise ValueError('Race updates must be strictly chronological and unique.')
        n = len(race)
        added = []
        for _, r in race.iterrows():
            if r['Did Not Start']:
                continue
            clean_finish = not (r['Did Not Finish'] or r['Did Not Start'] or r['Disqualified'])
            mates = race[(race['Team ID'] == r['Team ID']) & (race['Driver Abbr'] != r['Driver Abbr'])]
            mate_q = pd.to_numeric(mates['Quali Position'], errors='coerce').mean()
            grid = finite_float(r['Grid Position'])
            pos, q = finite_float(r['Race Position']), finite_float(r['Quali Position'])
            added.append((r['Driver Abbr'], {
                'year': year, 'round': rnd, 'team': r['Team ID'], 'circuit': r['Circuit ID'],
                'finish': scaled_position(pos, n) if clean_finish else np.nan,
                'quali': scaled_position(q, n), 'dnf': float(r['Did Not Finish']),
                'conversion': float(pos <= grid) if clean_finish and grid > 0 else np.nan,
                'mate_delta': (q - mate_q) / max(1, n - 1),
            }))
        for code, row in added:
            self.drivers[code].append(row)
        # Team history contains one mean per weekend, not an arbitrary ten cars.
        for tid in race['Team ID'].unique():
            team = [h for _, h in added if h['team'] == tid]
            def average(k):
                values = [h[k] for h in team if np.isfinite(h[k])]
                return float(np.mean(values)) if values else np.nan
            self.teams[tid].append({'year': year, 'finish': average('finish'),
                                    'quali': average('quali'), 'dnf': average('dnf')})
        for _, r in constructor_round.iterrows():
            self.standings[(year, r['Team ID'])] = {
                'rank_pct': (float(r['Rank']) - 1) / max(1, constructor_round['Rank'].max() - 1),
                'points_per_race': float(r['Total Points']) / rnd,
            }
        self.last_key = (year, rnd)

def make_training_frame(drivers, constructors, window=7, season_discount=.4):
    validate_pair(drivers, constructors)
    builder = FeatureBuilder(window, season_discount)
    frames = []
    for (year, rnd), race in drivers.groupby(['Year', 'Round Number'], sort=True):
        full_race = race
        # Retain classified DNS entries: removing them using race outcomes would
        # give a retrospective forecast information it did not have pre-race.
        race = race[race['Race Position'].notna()].sort_values('Driver Abbr').reset_index(drop=True)
        frame = builder.make_features(race, year, rnd, race.iloc[0]['Circuit ID'])
        frame['Year'], frame['Round Number'] = int(year), int(rnd)
        frame['Race ID'] = f'{int(year)}-{int(rnd):02d}'
        frame['Driver Abbr'] = race['Driver Abbr'].values
        for key in ['Race Position', 'Did Not Finish', 'Did Not Start', 'Disqualified']:
            frame[key] = race[key].values
        frames.append(frame)
        ct = constructors[(constructors.Year == year) & (constructors['Round Number'] == rnd)]
        builder.observe(full_race, ct)
    return pd.concat(frames, ignore_index=True)

def make_prediction_features(drivers, constructors, entries, year, rnd, circuit_id, window=7, season_discount=.4):
    builder = FeatureBuilder(window, season_discount)
    for (y, r), race in drivers.groupby(['Year', 'Round Number'], sort=True):
        if (int(y), int(r)) >= (int(year), int(rnd)):
            break
        ct = constructors[(constructors.Year == y) & (constructors['Round Number'] == r)]
        builder.observe(race, ct)
    return builder.make_features(entries, year, rnd, circuit_id)


# Executed after the original feature definitions in the self-contained notebook.
BASE_NUMERIC_FEATURES = list(NUMERIC_FEATURES)
BaseFeatureBuilder = FeatureBuilder
STRENGTH_FEATURES = [
    'driver_recent_q1_gap', 'team_recent_q1_gap', 'driver_season_q1_gap',
    'team_season_q1_gap', 'team_pace_change', 'current_team_q1_gap',
    'current_teammate_q1_delta', 'driver_season_finish', 'team_season_finish',
]
PRACTICE_FEATURES = FP_COLUMNS + ['fp_team_long_gap', 'fp_teammate_long_delta', 'fp_available']


def set_feature_group(group):
    global FEATURE_GROUP, FEATURE_VERSION, NUMERIC_FEATURES, FEATURE_COLUMNS
    if group not in ('baseline', 'strength', 'practice'):
        raise ValueError(f'Unknown feature group: {group}')
    FEATURE_GROUP = group
    FEATURE_VERSION = 'f1-fastf1-v3-' + group
    NUMERIC_FEATURES = BASE_NUMERIC_FEATURES + (STRENGTH_FEATURES if group != 'baseline' else [])
    if group == 'practice':
        NUMERIC_FEATURES += PRACTICE_FEATURES
    FEATURE_COLUMNS = NUMERIC_FEATURES + CATEGORICAL_FEATURES


class FeatureBuilder(BaseFeatureBuilder):
    def make_features(self, entries, year, rnd, circuit_id):
        frame = super().make_features(entries, year, rnd, circuit_id)
        if FEATURE_GROUP == 'baseline':
            return frame
        rows = []
        for _, entry in entries.iterrows():
            code, team = str(entry['Driver Abbr']), str(entry['Team ID'])
            dh, th = self.drivers[code], self.teams[team]
            def history_mean(history, key, default=np.nan):
                return weighted_history(history, key, year, self.window, default, self.season_discount)
            def season_mean(history, key, season, default=np.nan):
                values = [h[key] for h in history if h['year'] == season and np.isfinite(h.get(key, np.nan))]
                return float(np.mean(values)) if values else default
            # A finite fallback is needed inside weighted_history's confidence blend.
            tq = history_mean(th, 'q1_gap_pct', 0.)
            ts = season_mean(th, 'q1_gap_pct', year, tq)
            current_team = entries[entries['Team ID'] == team]
            mates = current_team[current_team['Driver Abbr'] != code]
            mate_q1 = pd.to_numeric(mates['Q1 Gap Percent'], errors='coerce').mean()
            row = {
                'driver_recent_q1_gap': history_mean(dh, 'q1_gap_pct', tq),
                'team_recent_q1_gap': tq,
                'driver_season_q1_gap': season_mean(dh, 'q1_gap_pct', year, ts),
                'team_season_q1_gap': ts,
                'team_pace_change': ts - season_mean(th, 'q1_gap_pct', year-1, ts),
                'current_team_q1_gap': pd.to_numeric(current_team['Q1 Gap Percent'], errors='coerce').mean(),
                'current_teammate_q1_delta': finite_float(entry.get('Q1 Gap Percent')) - mate_q1,
                'driver_season_finish': season_mean(dh, 'finish', year, frame.iloc[len(rows)].driver_form),
                'team_season_finish': season_mean(th, 'finish', year, frame.iloc[len(rows)].team_form),
            }
            if FEATURE_GROUP == 'practice':
                row.update({key: finite_float(entry.get(key)) for key in FP_COLUMNS})
                team_pace = pd.to_numeric(current_team.get('fp_long_gap', pd.Series(dtype=float)), errors='coerce').mean()
                mate_pace = pd.to_numeric(mates.get('fp_long_gap', pd.Series(dtype=float)), errors='coerce').mean()
                row['fp_team_long_gap'] = team_pace
                row['fp_teammate_long_delta'] = row['fp_long_gap'] - mate_pace
                row['fp_available'] = float(np.isfinite(row['fp_short_gap']) or np.isfinite(row['fp_long_gap']))
            rows.append(row)
        extra = pd.DataFrame(rows)
        for key in extra:
            frame[key] = extra[key].to_numpy()
        return frame[FEATURE_COLUMNS]

    def observe(self, race, constructor_round):
        super().observe(race, constructor_round)
        if FEATURE_GROUP == 'baseline':
            return
        for _, row in race.iterrows():
            if row['Did Not Start']:
                continue
            self.drivers[str(row['Driver Abbr'])][-1]['q1_gap_pct'] = finite_float(row.get('Q1 Gap Percent'))
        for tid, group in race.groupby('Team ID'):
            self.teams[str(tid)][-1]['q1_gap_pct'] = pd.to_numeric(group['Q1 Gap Percent'], errors='coerce').mean()


set_feature_group('baseline')


import importlib.metadata
import joblib
from scipy.stats import spearmanr
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import brier_score_loss, log_loss
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from xgboost import XGBRanker

SEED = 42
ORDER_POLICY_VERSION = 'fastf1-feature-groups-v3'

def new_preprocessor():
    return ColumnTransformer([
        ('numeric', Pipeline([
            ('impute', SimpleImputer(strategy='median', add_indicator=True, keep_empty_features=True)),
            ('scale', StandardScaler()),
        ]), NUMERIC_FEATURES),
        ('category', OneHotEncoder(handle_unknown='ignore', sparse_output=False), CATEGORICAL_FEATURES),
    ], sparse_threshold=0.)

def fit_models(train, depth=3, trees=180):
    train = train.sort_values(['Race ID', 'Driver Abbr']).reset_index(drop=True)
    pre = new_preprocessor()
    X = pre.fit_transform(train[FEATURE_COLUMNS])
    clean = ((train['Did Not Finish'] == 0) & (train['Did Not Start'] == 0) & (train['Disqualified'] == 0)).to_numpy()
    ranker = XGBRanker(
        objective='rank:pairwise', n_estimators=trees, max_depth=depth,
        learning_rate=.035, min_child_weight=8, reg_lambda=12.,
        subsample=.9, colsample_bytree=.9, tree_method='hist',
        random_state=SEED, n_jobs=2,
    )
    qids = pd.factorize(train.loc[clean, 'Race ID'], sort=True)[0]
    relevance = train.loc[clean, 'field_size'].to_numpy() - train.loc[clean, 'Race Position'].to_numpy()
    ranker.fit(X[clean], relevance, qid=qids)
    reliability = ((train['Did Not Start'] == 0) & (train['Disqualified'] == 0)).to_numpy()
    y_dnf = train.loc[reliability, 'Did Not Finish'].astype(int).to_numpy()
    prior = float((y_dnf.sum() + 2) / (len(y_dnf) + 20))
    if len(np.unique(y_dnf)) == 2:
        # No balanced class weights: they distort probabilities at the natural DNF prevalence.
        classifier = LogisticRegression(C=.05, max_iter=1500, random_state=SEED)
        classifier.fit(X[reliability], y_dnf)
    else:
        classifier = None
    return {'preprocessor': pre, 'ranker': ranker, 'dnf_model': classifier,
            'dnf_prior': prior, 'depth': depth, 'trees': trees}

def raw_model_predictions(bundle, frame):
    X = bundle['preprocessor'].transform(frame[FEATURE_COLUMNS])
    score = bundle['ranker'].predict(X)
    probability = (bundle['dnf_model'].predict_proba(X)[:, 1] if bundle['dnf_model'] is not None
                   else np.full(len(frame), bundle['dnf_prior']))
    # The shrinkage strength is chosen solely on pre-test validation races.
    shrink = bundle.get('dnf_shrink', 1.)
    probability = shrink * probability + (1 - shrink) * bundle['dnf_prior']
    return score, np.clip(probability, .001, .999)

def effective_grid_weight(race, candidate):
    """Attenuate only the extra blend; actual grid remains a ranker input.

    Displacement is observed qualifying-to-grid movement, not a verified penalty.
    Missing qualifying and pit-lane starts retain the fixed blend.
    """
    weight = np.full(len(race), float(candidate.get('grid_weight', 0.)))
    gate = candidate.get('grid_gate', 'none')
    if gate == 'none':
        return weight
    if gate not in ('displacement', 'pace_supported'):
        raise ValueError(f'Unknown grid gate: {gate}')
    valid = ((race['quali_missing'].to_numpy() == 0) &
             (race['pit_lane_start'].to_numpy() == 0) &
             (race['grid_missing'].to_numpy() == 0))
    loss = np.nan_to_num(race['grid_penalty'].to_numpy(), nan=0.)
    attenuation = np.clip(loss / .25, 0., 1.)
    if gate == 'pace_supported':
        advantage = race['grid_pct'].to_numpy() - race['expected_pace'].to_numpy()
        attenuation *= np.clip(np.nan_to_num(advantage, nan=0.) / .25, 0., 1.)
    return weight * (1. - np.where(valid, attenuation, 0.))


def order_cost(frame, scores, probabilities, candidate):
    costs = pd.Series(index=frame.index, dtype=float)
    for _, race in frame.groupby('Race ID', sort=False):
        idx = race.index
        if candidate['kind'] == 'grid':
            pace = race['grid_pct'].to_numpy()
        elif candidate['kind'] == 'form_blend':
            pace = .75 * race['grid_pct'].to_numpy() + .25 * race['expected_pace'].to_numpy()
        else:
            ml_rank = pd.Series(-np.asarray(scores)[idx]).rank(method='average').to_numpy()
            ml_pct = (ml_rank - 1) / max(1, len(race) - 1)
            grid_weight = effective_grid_weight(race, candidate)
            pace = grid_weight * race['grid_pct'].to_numpy() + (1 - grid_weight) * ml_pct
        # A validated expected-rank penalty, not an unconditional push toward the back.
        risk = candidate.get('risk_weight', 0.)
        costs.loc[idx] = pace + risk * np.asarray(probabilities)[idx] * (1. - pace)
    return costs.to_numpy()

def predict_race(bundle, drivers, constructors, entries, year, rnd, circuit_id):
    requested = f'{int(year)}-{int(rnd):02d}'
    if bundle['last_training_race'] >= requested:
        raise ValueError('This model was trained on this race or later races. Use an earlier checkpoint for a backtest.')
    if bundle['feature_version'] != FEATURE_VERSION or bundle['feature_columns'] != FEATURE_COLUMNS:
        raise ValueError('Feature schema/version does not match the saved model.')
    if bundle.get('order_policy_version') != ORDER_POLICY_VERSION:
        raise ValueError('Load or train a v3 model; this checkpoint has a different ordering policy.')
    recipe = bundle['recipe']
    features = make_prediction_features(drivers, constructors, entries, year, rnd, circuit_id,
                                       recipe['window'], recipe['season_discount']).reset_index(drop=True)
    features['Race ID'] = requested
    score, probability = raw_model_predictions(bundle, features)
    cost = order_cost(features, score, probability, recipe)
    result = pd.DataFrame({'Driver': entries['Driver Abbr'].to_numpy(),
                          'Starting Grid': entries['Grid Position'].to_numpy(),
                          'Qualifying Position': entries['Quali Position'].to_numpy(),
                          'Grid Displacement': np.where(entries['Grid Position'].to_numpy() > 0,
                              entries['Grid Position'].to_numpy() - entries['Quali Position'].to_numpy(), np.nan),
                          'Effective Grid Weight': effective_grid_weight(features, recipe),
                          'Order Score': cost, 'DNF Probability': probability})
    result['_grid_tie'] = features.grid_pct.to_numpy()
    result = result.sort_values(['Order Score', '_grid_tie', 'Driver']).reset_index(drop=True)
    result['Predicted Rank'] = np.arange(1, len(result) + 1)
    return result[['Predicted Rank', 'Driver', 'Starting Grid', 'Qualifying Position',
                   'Grid Displacement', 'Effective Grid Weight', 'Order Score', 'DNF Probability']]

from datetime import datetime, timezone
import uuid

# Frozen baseline recipe used by the verified walk-forward comparison.
# Weekly operation refits weights; it never repeats hyperparameter selection.
WEEKLY_RECIPE = {
    'feature_group': 'baseline', 'window': 4, 'season_discount': .25,
    'depth': 4, 'trees': 180, 'dnf_shrink': .5,
    'name': 'ranker_grid0.65_risk0_displacement', 'kind': 'ranker',
    'grid_weight': .65, 'risk_weight': 0., 'grid_gate': 'displacement',
}
WEEKLY_VERSION = 'baseline-weekly-v1'
# v1.1 changes qualifying acquisition only. The baseline recipe, feature
# builder, preprocessing and fitted estimators are unchanged from this hash.
COMPATIBLE_RUNTIME_HASHES = {'bda0702ea8b5023caf335aeaa801860861ad52ab10d78637f934c3d7e1e7386f'}
COMPATIBLE_RUNTIME_HASHES.add('7b79b5055bf65bdad851ced9152730d0216f4264d838401ac304c2f6fd5ab3bd')


def atomic_json(path, payload):
    path = Path(path)
    tmp = path.with_name(path.name + '.tmp')
    tmp.write_text(json.dumps(payload, indent=2, default=str))
    os.replace(tmp, path)


def cutoff_data(drivers, constructors, cutoff):
    def before(frame):
        return ((frame.Year < cutoff[0]) |
                ((frame.Year == cutoff[0]) & (frame['Round Number'] <= cutoff[1])))
    d = drivers.loc[before(drivers)].copy().sort_values(DRIVER_KEY).reset_index(drop=True)
    t = constructors.loc[before(constructors)].copy().sort_values(TEAM_KEY).reset_index(drop=True)
    validate_pair(d, t)
    available = set(map(tuple, d[['Year', 'Round Number']].drop_duplicates().to_numpy()))
    missing = [(cutoff[0], r) for r in range(1, cutoff[1] + 1)
               if (cutoff[0], r) not in available]
    if missing:
        raise ValueError(f'Completed results are missing: {missing}. Set UPDATE_WEEKEND=True.')
    for year, frame in d.groupby('Year'):
        rounds = set(frame['Round Number'])
        if rounds != set(range(1, int(max(rounds)) + 1)):
            raise ValueError(f'Historical round gaps in {year}; repair the CSV before training.')
    return d, t


def check_loaded_model(bundle, fingerprints, cutoff):
    expected = f'{cutoff[0]}-{cutoff[1]:02d}'
    if bundle.get('weekly_version') != WEEKLY_VERSION:
        raise ValueError('No compatible weekly baseline model. Set TRAIN_MODEL=True.')
    if bundle.get('recipe') != WEEKLY_RECIPE:
        raise ValueError('Saved recipe differs from the fixed baseline. Set TRAIN_MODEL=True.')
    if bundle.get('last_training_race') != expected:
        raise ValueError(f'Model trained through {bundle.get("last_training_race")}, expected {expected}. Set TRAIN_MODEL=True.')
    if bundle.get('training_fingerprints') != fingerprints:
        raise ValueError('CSV history changed since training. Set TRAIN_MODEL=True.')
    if bundle.get('runtime_sha256') not in {RUNTIME_SHA256, *COMPATIBLE_RUNTIME_HASHES}:
        raise ValueError('Model code changed since training. Set TRAIN_MODEL=True.')
    if bundle.get('versions') != package_versions():
        raise ValueError('Package versions changed since training. Set TRAIN_MODEL=True.')


def package_versions():
    return {p: importlib.metadata.version(p) for p in
            ['numpy', 'pandas', 'scipy', 'scikit-learn', 'xgboost', 'joblib']}


def qualifying_rows(order, times, drivers, overrides):
    if not 15 <= len(order) <= 24 or len(order) != len(set(order)):
        raise ValueError('QUALIFYING_ORDER needs 15–24 unique abbreviations in qualifying order, separately from grid penalties.')
    rows = []
    for position, code in enumerate(order, 1):
        past = drivers[drivers['Driver Abbr'] == code].sort_values(['Year', 'Round Number'])
        team = overrides.get(code) or (past.iloc[-1]['Team ID'] if len(past) else None)
        if team is None:
            raise ValueError(f'Supply TEAM_OVERRIDES for new driver {code}.')
        row = {'position': str(position), 'Driver': {'code': code},
               'Constructor': {'constructorId': team}}
        for segment in ['Q1', 'Q2', 'Q3']:
            value = times.get(code, {}).get(segment)
            if value is not None and str(value).strip():
                if not np.isfinite(seconds(value)) or seconds(value) <= 0:
                    raise ValueError(f'Invalid qualifying time: {code} {segment} {value}')
                row[segment] = str(value)
        rows.append(row)
    return rows


def get_weekly_qualifying(config, target, drivers, cache_dir):
    source = config.get('QUALIFYING_SOURCE', 'auto').lower()
    if source not in ['auto', 'manual']:
        raise ValueError('QUALIFYING_SOURCE must be auto or manual.')
    if source == 'manual':
        order = [str(code).strip().upper() for code in config.get('QUALIFYING_ORDER', [])]
        circuit_id = str(config.get('MANUAL_CIRCUIT_ID', '')).strip()
        if not circuit_id:
            raise ValueError('Set MANUAL_CIRCUIT_ID for manual qualifying. Use its historical API ID for an existing track.')
        rows = qualifying_rows(order, config.get('QUALIFYING_TIMES', {}), drivers,
                               config.get('TEAM_OVERRIDES', {}))
        print('Using explicitly supplied qualifying order. Team IDs come from latest history unless overridden.')
        print(f'Qualifying time coverage: {sum("Q1" in r for r in rows)}/{len(rows)} Q1; missing gaps use the trained imputer.')
        return {'season': str(target[0]), 'round': str(target[1]),
                'raceName': config.get('MANUAL_EVENT_NAME') or circuit_id,
                'Circuit': {'circuitId': circuit_id, 'circuitName': circuit_id,
                            'Location': {'locality': config.get('MANUAL_EVENT_NAME') or circuit_id}},
                'QualifyingResults': rows, '_source': 'user_manual',
                '_missing_times': not any('Q1' in r for r in rows)}
    try:
        q = public_race_session(*target, 'qualifying', cache_dir, refresh=True)
    except Exception as exc:
        print('Primary qualifying API failed:', type(exc).__name__, str(exc))
        q = None
    if q and q.get('QualifyingResults'):
        q['_source'] = 'jolpica'
        return q
    print('Primary qualifying results unavailable; trying FastF1 qualifying timing.')
    try:
        import fastf1
        ff_cache = Path(cache_dir) / 'fastf1'; ff_cache.mkdir(parents=True, exist_ok=True)
        fastf1.Cache.enable_cache(str(ff_cache))
        session = fastf1.get_session(*target, 'Q')
        if int(session.event['RoundNumber']) != target[1]:
            raise ValueError('FastF1 returned a different round.')
        session.load(laps=True, telemetry=False, weather=False, messages=True)
        result = session.results.sort_values('Position')
        positions = pd.to_numeric(result.Position, errors='coerce')
        if (positions.isna().any() or sorted(positions.tolist()) != list(range(1, len(result) + 1))
                or result['Q3'].isna().all()):
            raise ValueError('FastF1 qualifying order is incomplete; use confirmed manual qualifying.')
        order = result.Abbreviation.tolist()
        times = {r.Abbreviation: {s: str(seconds(r[s])) for s in ['Q1', 'Q2', 'Q3']
                                  if pd.notna(r[s])} for _, r in result.iterrows()}
        teams = dict(config.get('TEAM_OVERRIDES', {}))
        for _, row in result.iterrows():
            if pd.notna(row.get('TeamId')) and str(row.get('TeamId')).strip():
                teams.setdefault(row.Abbreviation, str(row.TeamId))
        rows = qualifying_rows(order, times, drivers, teams)
        schedule = public_race_session(*target, 'schedule', cache_dir, refresh=True)
        if not schedule or 'Circuit' not in schedule:
            raise ValueError('Circuit metadata unavailable. Use manual mode with an explicit circuit ID.')
        schedule['QualifyingResults'] = rows
        schedule['_source'] = 'fastf1_qualifying_timing'
        print('FastF1 timing-derived qualifying loaded; verify classification if penalties were applied.')
        return schedule
    except Exception as exc:
        raise ValueError('Automatic qualifying unavailable. Set QUALIFYING_SOURCE="manual" and supply QUALIFYING_ORDER and MANUAL_CIRCUIT_ID. The saved model is retained. FastF1 reason: ' + str(exc)) from exc


def openf1_read(endpoint, params):
    from urllib.parse import urlencode
    # Never fetch race finishing results for a pre-race prediction.
    if endpoint not in {'sessions', 'drivers', 'starting_grid'}:
        raise ValueError('Unsupported pre-race endpoint.')
    headers = {'User-Agent': 'F1-weekly-baseline/1.2'}
    token = os.environ.get('OPENF1_API_TOKEN')
    if token:
        headers['Authorization'] = 'Bearer ' + token
    request = urllib.request.Request('https://api.openf1.org/v1/' + endpoint + '?' + urlencode(params), headers=headers)
    with urllib.request.urlopen(request, timeout=30) as response:
        payload = json.load(response)
    if not isinstance(payload, list):
        raise ValueError(f'Unexpected OpenF1 {endpoint} response.')
    return payload


def published_starting_grid(qualifying, target):
    # Join by an explicit scheduled race timestamp, never by 'latest'.
    if not qualifying.get('date') or not qualifying.get('time'):
        raise ValueError('Scheduled race date/time unavailable for matching the grid API.')
    expected = pd.to_datetime(qualifying['date'] + 'T' + qualifying['time'], utc=True)
    sessions = openf1_read('sessions', {'year': target[0], 'session_name': 'Race'})
    matches = [s for s in sessions if s.get('session_name') == 'Race'
               and int(s.get('year', 0)) == target[0] and not s.get('is_cancelled', False)
               and abs(pd.to_datetime(s['date_start'], utc=True) - expected) <= pd.Timedelta(hours=6)]
    if len(matches) != 1:
        raise ValueError('Could not uniquely match the scheduled event to an OpenF1 race session.')
    session = matches[0]
    sk = int(session['session_key'])
    raw = openf1_read('starting_grid', {'session_key': sk})
    numbers = {int(r['number']): str(r['Driver']['code']).strip().upper()
               for r in qualifying['QualifyingResults'] if r.get('number') is not None}
    required_numbers = {int(row['driver_number']) for row in raw}
    people = (openf1_read('drivers', {'session_key': sk})
              if not required_numbers.issubset(numbers) else [])
    if not raw:
        raise ValueError('Published starting grid is not available yet.')
    if any(int(row.get('session_key', -1)) != sk for row in raw + people):
        raise ValueError('Grid/driver response contains a different session.')
    for person in people:
        number = int(person['driver_number'])
        code = str(person['name_acronym']).strip().upper()
        if number in numbers and numbers[number] != code:
            raise ValueError('Ambiguous driver-number mapping.')
        numbers[number] = code
    grid_positions = {}
    for row in raw:
        code = numbers.get(int(row['driver_number']))
        value = row.get('position')
        if not code or code in grid_positions or value is None:
            raise ValueError('Grid has an unmapped, duplicate or unclassified driver; use manual mode.')
        position = float(value)
        if not np.isfinite(position) or position < 0 or position % 1:
            raise ValueError('Grid position is ambiguous; use manual mode with explicit pit-lane starters.')
        grid_positions[code] = int(position)
    positive = [p for p in grid_positions.values() if p > 0]
    if len(positive) != len(set(positive)) or max(positive, default=0) > 24:
        raise ValueError('Published grid positions are invalid.')
    if not 15 <= len(grid_positions) <= 24:
        raise ValueError('Published grid is incomplete.')
    grid = sorted(grid_positions, key=lambda c: (grid_positions[c] == 0, grid_positions[c], c))
    return grid, grid_positions, {'source': 'openf1_starting_grid', 'session': session, 'rows': raw, 'drivers': people}


def resolve_prediction_inputs(config, target, drivers, cache_dir):
    mode = config['INPUT_MODE'].lower()
    if mode == 'manual':
        # A blank qualifying order explicitly means "use entered grid as a
        # qualifying proxy"; this is logged and never described as API data.
        manual = dict(config, QUALIFYING_SOURCE='manual')
        assumed = not bool(config.get('QUALIFYING_ORDER'))
        if assumed:
            manual['QUALIFYING_ORDER'] = list(config['STARTING_GRID'])
            print('Manual mode: assuming qualifying order equals your entered grid. Enter QUALIFYING_ORDER separately for penalty/custom-grid experiments.')
        q = get_weekly_qualifying(manual, target, drivers, cache_dir)
        q['_grid_used_as_qualifying_proxy'] = assumed
        return q, None, None, {'source': 'manual_grid', 'qualifying_assumed_from_grid': assumed}
    try:
        auto = dict(config, QUALIFYING_SOURCE='auto', TEAM_OVERRIDES={})
        q = get_weekly_qualifying(auto, target, drivers, cache_dir)
        grid, positions, audit = published_starting_grid(q, target)
        return q, grid, positions, audit
    except Exception as exc:
        raise ValueError('API inputs are not ready or could not be verified. Select INPUT_MODE="manual" and enter your grid; no automatic substitution was made. Reason: ' + str(exc)) from exc


def run_weekly(config):
    cutoff = (int(config['COMPLETED_YEAR']), int(config['COMPLETED_ROUND']))
    target = (int(config['PREDICT_YEAR']), int(config['PREDICT_ROUND']))
    update, train, predict = (config[k] for k in
                             ['UPDATE_WEEKEND', 'TRAIN_MODEL', 'RUN_PREDICTION'])
    mode = str(config.get('INPUT_MODE', 'manual')).lower()
    if mode not in ['api', 'manual']:
        raise ValueError('INPUT_MODE must be api or manual.')
    config = dict(config, INPUT_MODE=mode)
    if not all(isinstance(v, bool) for v in [update, train, predict]):
        raise ValueError('UPDATE_WEEKEND, TRAIN_MODEL and RUN_PREDICTION must be True or False.')
    if not any([update, train, predict]):
        print('All switches are False. Nothing requested.')
        return None
    if cutoff[1] < 1:
        raise ValueError('COMPLETED_ROUND must be the latest completed race round, at least 1.')
    grid = [str(v).strip().upper() for v in config['STARTING_GRID']]
    pit_lane = [str(v).strip().upper() for v in config.get('PIT_LANE_STARTERS', [])]
    withdrawals = set(config.get('WITHDRAWN_DRIVERS', []))
    if predict:
        if not (target == (cutoff[0], cutoff[1] + 1) or
                (target[0] == cutoff[0] + 1 and target[1] == 1)):
            raise ValueError('Predict the next round after COMPLETED_ROUND (or next season R1).')
        if mode == 'manual' and (not 15 <= len(grid) <= 24 or len(grid) != len(set(grid))):
            raise ValueError('Fill STARTING_GRID with 15–24 unique abbreviations, P1 first. Or set RUN_PREDICTION=False to update/train only.')
        if mode == 'manual' and (not set(pit_lane).issubset(grid) or len(pit_lane) != len(set(pit_lane))):
            raise ValueError('PIT_LANE_STARTERS must be unique drivers included in STARTING_GRID.')
        if mode == 'manual' and withdrawals.intersection(grid):
            raise ValueError('Withdrawn drivers must be excluded from STARTING_GRID.')

    data_dir = Path(config['DATA_DIR'])
    driver_path = data_dir / 'driver_f1new_corrected.csv'
    team_path = data_dir / 'constructors_f1new_corrected.csv'
    for path in [driver_path, team_path]:
        if not path.exists():
            raise FileNotFoundError(f'Missing input: {path}')
    drivers, constructors = pd.read_csv(driver_path), pd.read_csv(team_path)
    validate_pair(drivers, constructors)
    home = data_dir / 'weekly_baseline'
    home.mkdir(parents=True, exist_ok=True)
    run_id = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '_' + uuid.uuid4().hex[:8]
    run_dir = home / 'runs' / run_id
    run_dir.mkdir(parents=True)
    manifest = {'status': 'running', 'config': config, 'started_at_utc': run_id,
                'weekly_version': WEEKLY_VERSION, 'runtime_sha256': RUNTIME_SHA256,
                'recipe': WEEKLY_RECIPE, 'versions': package_versions()}
    atomic_json(run_dir / 'manifest.json', manifest)
    cache_dir = home / 'public_cache'
    model_path = home / 'baseline_model.joblib'
    try:
        if update:
            existing = set(drivers.loc[drivers.Year == cutoff[0], 'Round Number'])
            pending = sorted(set(range(1, cutoff[1] + 1)) - existing | {cutoff[1]})
            for rnd in pending:
                print(f'Updating completed results: {cutoff[0]} R{rnd:02d}', flush=True)
                drivers, constructors = append_single_race(
                    cutoff[0], rnd, driver_path, team_path, cache_dir, refresh=True)
        drivers, constructors = cutoff_data(drivers, constructors, cutoff)
        cutoff_id = f'{cutoff[0]}-{cutoff[1]:02d}'
        print(f'Available through {cutoff_id}: {drivers[DRIVER_KEY[:2]].drop_duplicates().shape[0]} races.', flush=True)
        last = drivers[(drivers.Year == cutoff[0]) & (drivers['Round Number'] == cutoff[1])]
        print('Latest completed event:', last['Race Name'].iloc[0])
        drivers.to_csv(run_dir / 'training_drivers.csv', index=False)
        constructors.to_csv(run_dir / 'training_constructors.csv', index=False)
        fingerprints = {'drivers': frame_fingerprint(drivers),
                        'constructors': frame_fingerprint(constructors)}
        manifest['training_fingerprints'] = fingerprints
        manifest['last_training_race'] = cutoff_id
        set_feature_group('baseline')
        if train:
            print('Training baseline v2 with fixed settings...', flush=True)
            frame = make_training_frame(drivers, constructors,
                                        WEEKLY_RECIPE['window'], WEEKLY_RECIPE['season_discount'])
            bundle = fit_models(frame, WEEKLY_RECIPE['depth'], WEEKLY_RECIPE['trees'])
            bundle.update({'recipe': dict(WEEKLY_RECIPE), 'dnf_shrink': WEEKLY_RECIPE['dnf_shrink'],
                           'feature_version': FEATURE_VERSION, 'feature_columns': list(FEATURE_COLUMNS),
                           'order_policy_version': ORDER_POLICY_VERSION, 'weekly_version': WEEKLY_VERSION,
                           'last_training_race': cutoff_id, 'training_fingerprints': fingerprints,
                           'runtime_sha256': RUNTIME_SHA256, 'versions': package_versions()})
            joblib.dump(bundle, run_dir / 'baseline_model.joblib')
            temp_model = home / 'baseline_model.tmp'
            shutil.copy2(run_dir / 'baseline_model.joblib', temp_model)
            os.replace(temp_model, model_path)
            print('Saved:', model_path, flush=True)
        elif predict:
            if not model_path.exists():
                raise FileNotFoundError('No weekly baseline model yet. Set TRAIN_MODEL=True for the first run.')
            bundle = joblib.load(model_path)
            check_loaded_model(bundle, fingerprints, cutoff)
            shutil.copy2(model_path, run_dir / 'baseline_model.joblib')
            print('Loaded baseline trained through:', bundle['last_training_race'])
        results = None
        if predict:
            print('Fetching qualifying for the prediction event...', flush=True)
            qualifying, api_grid, api_positions, grid_audit = resolve_prediction_inputs(config, target, drivers, cache_dir)
            if mode == 'api':
                grid = api_grid
                pit_lane = [code for code in grid if api_positions[code] == 0]
                # API mode ignores all hypothetical/manual driver settings.
                withdrawals = set()
            atomic_json(run_dir / 'grid_source.json', grid_audit)
            manifest['input_mode'] = mode
            manifest['grid_source'] = grid_audit['source']
            if not qualifying or not qualifying.get('QualifyingResults'):
                raise ValueError('Qualifying results are not available from the API yet. Saved training remains available; retry prediction later.')
            if (int(qualifying['season']), int(qualifying['round'])) != target:
                raise ValueError('Qualifying belongs to a different event.')
            circuit = qualifying['Circuit']
            location = circuit.get('Location', {}).get('locality', '')
            event_text = f'{qualifying["raceName"]} | {circuit["circuitName"]} | {location}'
            print('Prediction event:', event_text)
            manifest['qualifying_source'] = qualifying.get('_source', 'jolpica')
            expected = str(config.get('EXPECTED_PREDICTION_EVENT', '')).strip()
            if expected and expected.casefold() not in event_text.casefold():
                raise ValueError(f'Expected event/location containing {expected!r}; API returned {event_text}. Check the round.')
            qcodes = [r['Driver']['code'] for r in qualifying['QualifyingResults']]
            if len(qcodes) != len(set(qcodes)) or not withdrawals.issubset(qcodes):
                raise ValueError('Duplicate qualifying drivers or an invalid withdrawal list.')
            if set(grid) != set(qcodes) - withdrawals:
                raise ValueError(f'Grid differs from qualifying entrants. Missing: {sorted(set(qcodes)-withdrawals-set(grid))}; extra: {sorted(set(grid)-set(qcodes))}. Review STARTING_GRID/WITHDRAWN_DRIVERS.')
            overrides = {code: {'Team ID': tid} for code, tid in config.get('TEAM_OVERRIDES', {}).items()} if mode == 'manual' else {}
            if not set(overrides).issubset(grid):
                raise ValueError('TEAM_OVERRIDES contains a driver absent from the grid.')
            # Pit-lane drivers remain entrants but are not ordinary grid positions.
            normal_grid = [code for code in grid if code not in pit_lane]
            for code in grid:
                overrides.setdefault(code, {})['Grid Position'] = api_positions[code] if mode == 'api' else (0 if code in pit_lane else normal_grid.index(code) + 1)
            entries = build_entries(drivers, *target, grid, overrides, qualifying)
            circuit_id = circuit['circuitId']
            new_track = circuit_id not in set(drivers['Circuit ID'])
            if new_track:
                print('New circuit: using general driver/team form; no circuit history is invented.')
            results = predict_race(bundle, drivers, constructors, entries, *target, circuit_id)
            features = make_prediction_features(drivers, constructors, entries, *target, circuit_id,
                                                WEEKLY_RECIPE['window'], WEEKLY_RECIPE['season_discount'])
            entries.to_csv(run_dir / 'prediction_entries.csv', index=False)
            features.to_csv(run_dir / 'prediction_features.csv', index=False)
            results.to_csv(run_dir / 'prediction.csv', index=False)
            atomic_json(run_dir / 'qualifying.json', qualifying)
            manifest.update({'prediction_event': event_text, 'circuit_id': circuit_id,
                             'new_circuit': new_track, 'prediction_year': target[0], 'prediction_round': target[1],
                             'model_sha256': hashlib.sha256((run_dir / 'baseline_model.joblib').read_bytes()).hexdigest()})
            print(results.to_string(index=False))
            print('Prediction saved:', run_dir / 'prediction.csv')
        manifest['status'] = 'complete'
        manifest['finished_at_utc'] = datetime.now(timezone.utc).isoformat()
        atomic_json(run_dir / 'manifest.json', manifest)
        print('Run complete:', run_dir)
        return {'results': results, 'run_dir': run_dir, 'manifest': manifest}
    except Exception as exc:
        manifest.update({'status': 'failed', 'error': f'{type(exc).__name__}: {exc}'})
        atomic_json(run_dir / 'manifest.json', manifest)
        print('Stopped. Details and any completed work saved in:', run_dir)
        raise

RUNTIME_SHA256 = 'f0d0e2bd89076d70fdf2a3778561c56b7166a54700d4f0021d66fdf3c86c3ebc'


## Update → train/load → predict

In [ ]:
# Automatic workflow. No edits required.
weekly_config = {name: globals()[name] for name in [
    "DATA_DIR", "COMPLETED_YEAR", "COMPLETED_ROUND", "PREDICT_YEAR", "PREDICT_ROUND",
    "UPDATE_WEEKEND", "TRAIN_MODEL", "RUN_PREDICTION", "STARTING_GRID",
    "PIT_LANE_STARTERS", "WITHDRAWN_DRIVERS", "TEAM_OVERRIDES", "EXPECTED_PREDICTION_EVENT",
    "INPUT_MODE", "QUALIFYING_ORDER", "QUALIFYING_TIMES", "MANUAL_CIRCUIT_ID", "MANUAL_EVENT_NAME",
]}
weekly_run = run_weekly(weekly_config)


## Saved files

Everything is stored under `DATA_DIR/weekly_baseline/`:

- `baseline_model.joblib`: the current baseline checkpoint, trained only when requested.
- `runs/<timestamp>/prediction.csv`: predicted order and DNF probabilities.
- Each run also keeps input/history snapshots, qualifying, the actual model used and a manifest. Earlier run folders remain intact.

Prediction-only mode rejects a model whose cutoff, training data, code or package versions differ. Set `TRAIN_MODEL=True` to rebuild it. Failed prediction fetching does not discard a successfully trained model. Use prediction-only mode to retry after qualifying becomes available. Do not put the old automatically selected `f1_ranking_model.joblib` in the weekly model path.

Sources: [OpenF1 starting-grid API](https://openf1.org/docs/#starting-grid). Data access and publication timing are provider-dependent; real-time access may require an OpenF1 subscription. If you have authorized access, its bearer token can be supplied via the OPENF1_API_TOKEN environment variable; it is never saved in the run manifest. Manual mode needs no such access.

The recipe is fixed to the verified baseline comparison: window 4, season discount 0.25, depth 4, 180 trees, grid blend 0.65 with displacement adjustment, ranking DNF-risk weight 0, and DNF probability shrinkage 0.5. The DNF probability is reported separately; it does not add a risk penalty to the ranking under this recipe. This workflow refits weights without selecting new hyperparameters or rerunning backtests. Past results do not guarantee future accuracy.

Keep the lid open and the runtime connected during updates/training. If setup installs a missing dependency, follow its one-time restart instruction.
